# Local Level Model: The Foundation of State Space Forecasting

## Learning Objectives

By completing this notebook, you will:
1. Understand the local level model structure and its assumptions
2. Implement the local level model from scratch using Kalman filtering
3. Build Bayesian local level models in PyMC for commodity prices
4. Interpret filtered states, smoothed states, and forecast distributions
5. Compare local level forecasts with naive baselines
6. Diagnose model fit using innovation diagnostics

## Prerequisites
- Bayesian inference fundamentals
- Basic time series concepts (stationarity, autocorrelation)
- Linear algebra (matrix multiplication)

## Estimated Time: 90 minutes

---

In [ ]:
learning_objectives(["Understand the local level model structure and its assumptions", "Implement the local level model from scratch using Kalman filtering", "Build Bayesian local level models in PyMC for commodity prices", "Interpret filtered states, smoothed states, and forecast distributions", "Compare local level forecasts with naive baselines", "Diagnose model fit using innovation diagnostics"])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

apply_course_theme()
apply_plot_theme()

## 1. Setup and Imports

In [ ]:
# Core imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Bayesian stack
import pymc as pm
import arviz as az
from scipy import stats

# Time series
from statsmodels.tsa.statespace.structural import UnobservedComponents

# Configuration
np.random.seed(42)
az.style.use('arviz-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

print(f"PyMC version: {pm.__version__}")
print(f"ArviZ version: {az.__version__}")
print("Environment ready!")

## 2. What is a Local Level Model?

The **local level model** (also called "random walk plus noise") is the simplest state space model.

**Structure:**

$$\begin{align}
y_t &= \mu_t + \epsilon_t && \text{(Observation equation)} \\
\mu_t &= \mu_{t-1} + \eta_t && \text{(State equation)}
\end{align}$$

where:
- $y_t$ = observed price at time $t$
- $\mu_t$ = unobserved "true" level (latent state)
- $\epsilon_t \sim \mathcal{N}(0, \sigma_{\epsilon}^2)$ = observation noise
- $\eta_t \sim \mathcal{N}(0, \sigma_{\eta}^2)$ = state evolution noise

**Intuition:**
- The "true" price level $\mu_t$ follows a random walk
- We observe $\mu_t$ with measurement error $\epsilon_t$
- Goal: Filter out noise to estimate the underlying level

**Key Parameters:**
1. $\sigma_{\epsilon}^2$: Observation variance (how noisy are measurements?)
2. $\sigma_{\eta}^2$: State variance (how volatile is the true level?)
3. **Signal-to-noise ratio**: $q = \sigma_{\eta}^2 / \sigma_{\epsilon}^2$
   - Low $q$ → smooth estimates (trust measurements)
   - High $q$ → wiggly estimates (level changes quickly)

## 3. Simulate Data from Local Level Model

Let's generate synthetic commodity price data to understand the model.

In [ ]:
# Simulation parameters
np.random.seed(42)
T = 100  # Time periods

# True parameters
mu_0 = 75.0  # Initial level (e.g., oil price)
sigma_eta = 1.5  # State volatility (level changes gradually)
sigma_epsilon = 3.0  # Observation noise (noisy measurements)

# Generate latent states (random walk)
mu_true = np.zeros(T)
mu_true[0] = mu_0

for t in range(1, T):
    mu_true[t] = mu_true[t-1] + np.random.normal(0, sigma_eta)

# Generate observations (add measurement noise)
y_obs = mu_true + np.random.normal(0, sigma_epsilon, T)

# Create time index
time = np.arange(T)

print(f"Generated {T} observations")
print(f"True sigma_eta (state std): {sigma_eta}")
print(f"True sigma_epsilon (obs std): {sigma_epsilon}")
print(f"Signal-to-noise ratio q: {(sigma_eta**2 / sigma_epsilon**2):.3f}")

In [ ]:
# Visualize true state vs observations
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(time, mu_true, 'b-', linewidth=2, label='True Level (μ_t)', alpha=0.8)
ax.plot(time, y_obs, 'o', color='gray', markersize=4, alpha=0.5, label='Observations (y_t)')

ax.set_xlabel('Time', fontsize=12)
ax.set_ylabel('Price', fontsize=12)
ax.set_title('Local Level Model: True State vs Noisy Observations', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**What we see:**
- Blue line: Smooth underlying level (what we want to estimate)
- Gray dots: Noisy observations (what we actually see)
- Goal: Use Kalman filter to recover the blue line from gray dots

## 4. Kalman Filter from Scratch

The **Kalman filter** is the optimal recursive algorithm for estimating the hidden state $\mu_t$ from observations.

**Algorithm (for local level model):**

At each time $t$:

1. **Predict:**
   $$\begin{align}
   \mu_{t|t-1} &= \mu_{t-1|t-1} \\
   P_{t|t-1} &= P_{t-1|t-1} + \sigma_{\eta}^2
   \end{align}$$

2. **Update:**
   $$\begin{align}
   v_t &= y_t - \mu_{t|t-1} && \text{(Innovation)} \\
   S_t &= P_{t|t-1} + \sigma_{\epsilon}^2 && \text{(Innovation variance)} \\
   K_t &= P_{t|t-1} / S_t && \text{(Kalman gain)} \\
   \mu_{t|t} &= \mu_{t|t-1} + K_t v_t && \text{(Updated mean)} \\
   P_{t|t} &= (1 - K_t) P_{t|t-1} && \text{(Updated variance)}
   \end{align}$$

In [ ]:
def kalman_filter_local_level(y, sigma_epsilon, sigma_eta, mu_0=0, P_0=1e6):
    """
    Kalman filter for local level model.
    
    Parameters
    ----------
    y : array
        Observed time series
    sigma_epsilon : float
        Observation noise std
    sigma_eta : float
        State evolution noise std
    mu_0 : float
        Initial state mean
    P_0 : float
        Initial state variance
    
    Returns
    -------
    mu_filtered : array
        Filtered state means
    P_filtered : array
        Filtered state variances
    """
    T = len(y)
    
    # Storage
    mu_filtered = np.zeros(T)
    P_filtered = np.zeros(T)
    
    # Initialize
    mu_pred = mu_0
    P_pred = P_0
    
    for t in range(T):
        # Update step
        v = y[t] - mu_pred  # Innovation
        S = P_pred + sigma_epsilon**2  # Innovation variance
        K = P_pred / S  # Kalman gain
        
        mu_filt = mu_pred + K * v
        P_filt = (1 - K) * P_pred
        
        # Store
        mu_filtered[t] = mu_filt
        P_filtered[t] = P_filt
        
        # Predict next
        mu_pred = mu_filt
        P_pred = P_filt + sigma_eta**2
    
    return mu_filtered, P_filtered

# Test with true parameters
mu_filt, P_filt = kalman_filter_local_level(y_obs, sigma_epsilon, sigma_eta, 
                                             mu_0=y_obs[0], P_0=10)

print("Kalman filter complete!")
print(f"Final filtered state: {mu_filt[-1]:.2f}")
print(f"Final filtered variance: {P_filt[-1]:.2f}")

In [ ]:
# Plot filtered estimates vs truth
fig, ax = plt.subplots(figsize=(14, 6))

# Filtered estimate with uncertainty
std_filt = np.sqrt(P_filt)
ax.plot(time, mu_filt, 'r-', linewidth=2, label='Filtered Estimate', alpha=0.9)
ax.fill_between(time, mu_filt - 2*std_filt, mu_filt + 2*std_filt, 
                alpha=0.2, color='red', label='95% Confidence')

# Truth and observations
ax.plot(time, mu_true, 'b--', linewidth=2, label='True Level', alpha=0.7)
ax.plot(time, y_obs, 'o', color='gray', markersize=3, alpha=0.3, label='Observations')

ax.set_xlabel('Time', fontsize=12)
ax.set_ylabel('Price', fontsize=12)
ax.set_title('Kalman Filter: Recovering True Level from Noisy Data', fontsize=14)
ax.legend(fontsize=11, loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Key Insights:**

1. **Filtering works!** Red line closely tracks blue (true) line
2. **Uncertainty shrinks** over time as we accumulate data
3. **Smoothing**: Filter uses only past data; can be improved with full-sample "smoother"

## 5. Bayesian Local Level Model in PyMC

In practice, we don't know $\sigma_{\epsilon}$ and $\sigma_{\eta}$ - we must estimate them.

**Bayesian approach:**
- Place priors on both variance parameters
- Estimate latent states $\mu_t$ and parameters jointly via MCMC

In [ ]:
# Build Bayesian local level model
with pm.Model() as local_level_model:
    
    # Priors on variances
    # HalfNormal is good default for scale parameters
    sigma_epsilon = pm.HalfNormal('sigma_epsilon', sigma=5)
    sigma_eta = pm.HalfNormal('sigma_eta', sigma=2)
    
    # Initial state
    mu_0 = pm.Normal('mu_0', mu=y_obs[0], sigma=10)
    
    # State evolution: Random walk
    # PyMC's GaussianRandomWalk is designed for this
    mu = pm.GaussianRandomWalk('mu', mu=mu_0, sigma=sigma_eta, 
                               shape=T, init_dist=pm.Normal.dist(mu_0, sigma=0.1))
    
    # Observation model
    y_hat = pm.Normal('y_obs', mu=mu, sigma=sigma_epsilon, observed=y_obs)

# Visualize model structure
pm.model_to_graphviz(local_level_model)

In [ ]:
# Sample from posterior
with local_level_model:
    trace = pm.sample(
        draws=1000,
        tune=1000,
        chains=2,
        cores=1,
        random_seed=42,
        return_inferencedata=True
    )

print("\n✅ Sampling complete!")

In [ ]:
# Check parameter estimates
summary = az.summary(trace, var_names=['sigma_epsilon', 'sigma_eta', 'mu_0'])
print("Posterior Summary:")
print("="*70)
print(summary)
print("="*70)
print(f"\nTrue values:")
print(f"  sigma_epsilon: {sigma_epsilon}")
print(f"  sigma_eta: {sigma_eta}")
print(f"  mu_0: {mu_0}")

**Convergence Check:**
- `r_hat` close to 1.00 → chains converged
- `ess_bulk` > 400 → sufficient effective samples

In [ ]:
# Visualize posterior estimates of latent states
mu_posterior = trace.posterior['mu'].values  # Shape: (chains, draws, T)

# Compute posterior mean and credible intervals
mu_mean = mu_posterior.mean(axis=(0, 1))
mu_lower = np.percentile(mu_posterior, 2.5, axis=(0, 1))
mu_upper = np.percentile(mu_posterior, 97.5, axis=(0, 1))

fig, ax = plt.subplots(figsize=(14, 6))

# Posterior mean and credible interval
ax.plot(time, mu_mean, 'r-', linewidth=2, label='Posterior Mean State', alpha=0.9)
ax.fill_between(time, mu_lower, mu_upper, alpha=0.2, color='red', 
                label='95% Credible Interval')

# True state and observations
ax.plot(time, mu_true, 'b--', linewidth=2, label='True Level', alpha=0.7)
ax.plot(time, y_obs, 'o', color='gray', markersize=3, alpha=0.3, label='Observations')

ax.set_xlabel('Time', fontsize=12)
ax.set_ylabel('Price', fontsize=12)
ax.set_title('Bayesian Local Level Model: Estimated States', fontsize=14)
ax.legend(fontsize=11, loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Forecasting with Local Level Model

Generate **h-step-ahead forecasts** from the final state.

**Forecast distribution:**
$$y_{T+h} \sim \mathcal{N}(\mu_T, h \sigma_{\eta}^2 + \sigma_{\epsilon}^2)$$

Uncertainty grows linearly with horizon $h$.

In [ ]:
# Generate forecasts
h_max = 20  # Forecast horizon

# Extract posterior samples for final state and variances
mu_T = trace.posterior['mu'].values[:, :, -1].flatten()  # Final state
sigma_eps_samples = trace.posterior['sigma_epsilon'].values.flatten()
sigma_eta_samples = trace.posterior['sigma_eta'].values.flatten()

# Storage for forecasts
n_samples = len(mu_T)
forecasts = np.zeros((n_samples, h_max))

for i in range(n_samples):
    # Current state
    mu_current = mu_T[i]
    
    # Simulate forward
    for h in range(h_max):
        # State evolves
        mu_current = mu_current + np.random.normal(0, sigma_eta_samples[i])
        # Observation with noise
        y_forecast = mu_current + np.random.normal(0, sigma_eps_samples[i])
        forecasts[i, h] = y_forecast

# Compute forecast distribution statistics
forecast_mean = forecasts.mean(axis=0)
forecast_lower = np.percentile(forecasts, 2.5, axis=0)
forecast_upper = np.percentile(forecasts, 97.5, axis=0)

print(f"Generated {h_max}-step-ahead forecasts")

In [ ]:
# Plot forecasts
fig, ax = plt.subplots(figsize=(14, 6))

# Historical data
ax.plot(time, y_obs, 'o-', color='black', markersize=3, label='Observed Data', alpha=0.7)
ax.plot(time, mu_true, 'b--', linewidth=2, label='True Level', alpha=0.5)

# Forecasts
forecast_time = np.arange(T, T + h_max)
ax.plot(forecast_time, forecast_mean, 'r-', linewidth=2, label='Forecast Mean', alpha=0.9)
ax.fill_between(forecast_time, forecast_lower, forecast_upper, 
                alpha=0.3, color='red', label='95% Prediction Interval')

# Extend true level for comparison (if we generated it)
# In practice, we don't know future truth

ax.axvline(T, color='gray', linestyle='--', linewidth=1, alpha=0.5)
ax.text(T, ax.get_ylim()[1], ' Forecast Origin', fontsize=10, va='top')

ax.set_xlabel('Time', fontsize=12)
ax.set_ylabel('Price', fontsize=12)
ax.set_title('Local Level Model: Out-of-Sample Forecasts', fontsize=14)
ax.legend(fontsize=11, loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Key Features:**

1. **Flat forecast**: Local level assumes level continues as random walk (no trend)
2. **Widening uncertainty**: Prediction intervals grow over horizon
3. **Mean-reverting to final level**: Forecast mean ≈ last observed level

## 7. Model Diagnostics: Innovation Analysis

**Innovations** (one-step-ahead forecast errors) should be white noise if the model is well-specified.

$$v_t = y_t - \mathbb{E}[y_t | y_{1:t-1}]$$

**Tests:**
1. Plot innovations over time (look for patterns)
2. ACF of innovations (should be uncorrelated)
3. Normality test (Q-Q plot)

In [ ]:
# Compute one-step-ahead predictions and innovations
# Using filtered estimates

# For simplicity, use Kalman filter with posterior mean parameters
sigma_eps_mean = trace.posterior['sigma_epsilon'].mean().values
sigma_eta_mean = trace.posterior['sigma_eta'].mean().values

mu_filt_diag, P_filt_diag = kalman_filter_local_level(
    y_obs, sigma_eps_mean, sigma_eta_mean, mu_0=y_obs[0], P_0=10
)

# Compute one-step-ahead predictions (shift filtered by 1)
y_pred_onestep = np.concatenate([[y_obs[0]], mu_filt_diag[:-1]])

# Innovations
innovations = y_obs - y_pred_onestep

print(f"Computed {len(innovations)} innovations")
print(f"Innovation mean: {innovations.mean():.3f} (should be ≈ 0)")
print(f"Innovation std: {innovations.std():.3f}")

In [ ]:
# Diagnostic plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Innovations over time
axes[0, 0].plot(time, innovations, 'o-', alpha=0.6)
axes[0, 0].axhline(0, color='red', linestyle='--', linewidth=1)
axes[0, 0].set_xlabel('Time')
axes[0, 0].set_ylabel('Innovation')
axes[0, 0].set_title('Innovations Over Time (should be random)')
axes[0, 0].grid(True, alpha=0.3)

# 2. ACF of innovations
from statsmodels.graphics.tsaplots import plot_acf
plot_acf(innovations, lags=20, ax=axes[0, 1], alpha=0.05)
axes[0, 1].set_title('ACF of Innovations (should be uncorrelated)')

# 3. Histogram
axes[1, 0].hist(innovations, bins=30, density=True, alpha=0.7, edgecolor='black')
# Overlay normal
x_norm = np.linspace(innovations.min(), innovations.max(), 100)
axes[1, 0].plot(x_norm, stats.norm(0, innovations.std()).pdf(x_norm), 
                'r-', linewidth=2, label='Normal')
axes[1, 0].set_xlabel('Innovation')
axes[1, 0].set_ylabel('Density')
axes[1, 0].set_title('Histogram (should be approximately normal)')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 4. Q-Q plot
stats.probplot(innovations, dist="norm", plot=axes[1, 1])
axes[1, 1].set_title('Q-Q Plot (points should follow red line)')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Interpretation:**

1. **Time plot**: No obvious patterns → good
2. **ACF**: Most lags within confidence bands → uncorrelated → good
3. **Histogram**: Roughly normal → good
4. **Q-Q plot**: Points near line → normality holds → good

If diagnostics fail, consider:
- Adding trend component
- Allowing stochastic volatility
- Including external regressors

---

## Exercise 1: Signal-to-Noise Ratio Exploration

**Task:** Investigate how the signal-to-noise ratio $q = \sigma_{\eta}^2 / \sigma_{\epsilon}^2$ affects filtering.

1. Generate data with three different $q$ values: 0.1, 1.0, 10.0
2. Apply Kalman filter to each
3. Plot filtered estimates and compare smoothness

**Question:** When should you use a smoother vs more responsive filter?

In [ ]:
# YOUR CODE HERE
np.random.seed(42)

# Define scenarios
q_values = [0.1, 1.0, 10.0]
sigma_epsilon_ex1 = 3.0  # Fixed

# YOUR CODE: Generate data and filter for each q
# Store results for plotting

In [ ]:
# AUTO-GRADED TEST
def test_exercise_1():
    """Verify signal-to-noise exploration."""
    # Check that analysis was performed
    # (Test would check for existence of results)
    print("✅ Exercise 1 complete! Check your plots.")

# test_exercise_1()  # Uncomment after completing

---

## Exercise 2: Real Commodity Data

**Task:** Apply local level model to real crude oil prices.

1. Load WTI crude oil daily prices (use `yfinance` or similar)
2. Fit Bayesian local level model
3. Generate 30-day-ahead forecasts
4. Evaluate forecast accuracy if test data is available

**Hint:** Real commodity data often has heavier tails than Gaussian - this is a limitation.

In [ ]:
# YOUR CODE HERE
# Example starter:
# import yfinance as yf
# oil = yf.download('CL=F', start='2023-01-01', end='2024-01-01')
# prices = oil['Close'].values

# YOUR CODE: Fit local level model to real data

---

## Exercise 3: Forecast Evaluation

**Task:** Compare local level forecasts with naive benchmarks.

**Benchmarks:**
1. **Naive forecast**: $\hat{y}_{T+h} = y_T$ (last observation)
2. **Mean forecast**: $\hat{y}_{T+h} = \bar{y}$ (historical mean)

**Metric:** Mean Absolute Error (MAE) on held-out data

1. Split data into train (first 80%) and test (last 20%)
2. Fit local level on train
3. Generate forecasts for test period
4. Compute MAE and compare with benchmarks

In [ ]:
# YOUR CODE HERE

# Split data
split_point = int(0.8 * T)
y_train = y_obs[:split_point]
y_test = y_obs[split_point:]

# YOUR CODE: Fit model on y_train, forecast, compute MAE

In [ ]:
# AUTO-GRADED TEST
def test_exercise_3():
    """Verify forecast evaluation."""
    # Test would check MAE computations
    print("✅ Exercise 3 complete! Compare MAE across methods.")

# test_exercise_3()  # Uncomment after completing

---

## Exercise 4: Missing Data

**Task:** Handle missing observations using Kalman filter.

One advantage of state space models: they naturally handle missing data.

1. Take the original `y_obs` and randomly set 10% of values to NaN
2. Modify Kalman filter to skip update step when observation is missing
3. Show that filtered estimates still work

**Hint:** When $y_t$ is missing, only do prediction step (no update).

In [ ]:
# YOUR CODE HERE

# Create missing data
np.random.seed(42)
y_missing = y_obs.copy()
missing_idx = np.random.choice(T, size=int(0.1*T), replace=False)
y_missing[missing_idx] = np.nan

# YOUR CODE: Implement Kalman filter that handles NaN

---

## Solutions

### Exercise 1 Solution (Sketch)

In [ ]:
# SOLUTION
# for q in q_values:
#     sigma_eta_ex1 = np.sqrt(q * sigma_epsilon_ex1**2)
#     # Generate data with this q
#     # Apply Kalman filter
#     # Plot
#
# Observation:
# - Low q (0.1): Very smooth, slow to adapt
# - High q (10): Wiggly, quickly follows observations
# Use case: Smooth filter for reporting, responsive for trading signals

---

## Summary

### Key Takeaways

1. **Local level model** = random walk state + observation noise
2. **Kalman filter** optimally estimates hidden state from noisy observations
3. **Signal-to-noise ratio** controls smoothness of filtered estimates
4. **Bayesian approach** estimates variances from data, quantifies uncertainty
5. **Forecasts** from local level are flat with widening intervals
6. **Diagnostics** (innovations) reveal model misspecification

### Limitations of Local Level Model

- No trend (price level drifts randomly)
- No seasonality
- Gaussian errors (real commodities have fat tails)
- Constant variance (stochastic volatility often needed)

**Next:** Local linear trend model adds deterministic or stochastic trend.

### What's Next

Next notebook: **Local Linear Trend Model** - Adding trend to capture price momentum.

### Additional Resources

- Durbin & Koopman (2012): *Time Series Analysis by State Space Methods* (definitive reference)
- Harvey (1990): *Forecasting, Structural Time Series Models and the Kalman Filter*
- Petris et al. (2009): *Dynamic Linear Models with R*

---

*Remember: State space models separate signal from noise. The local level model is the foundation - master it before adding complexity.*